# Notebook 2: Estructuras para Agentes - POO, Tipos y Decoradores

## 🎯 Objetivo
Aprender las herramientas de Python que usarás **directamente** al construir agentes con LangChain/LangGraph.

## 🧠 ¿Por qué este contenido?
En frameworks de agentes trabajarás con:
- **Clases** → Los agentes, tools, y componentes son clases
- **Type hints** → LangGraph usa tipos para validar estados
- **Pydantic** → Para structured output (que el LLM devuelva datos estructurados)
- **Decoradores** → @tool, @chain y otros patrones comunes


## 📋 Contenido
1. **Clases y POO** → Entender cómo funcionan los componentes
2. **Type Hints** → Anotaciones de tipos en Python
3. **Pydantic** → Modelos para structured output
4. **Decoradores** → El patrón @tool

---
# PARTE 1: PROGRAMACIÓN ORIENTADA A OBJETOS (POO)

## ¿Qué es POO?
La **Programación Orientada a Objetos** organiza el código en **objetos** que contienen datos (**atributos**) y comportamiento (**métodos**).

## ¿Por qué necesitas POO para agentes?
En LangChain/LangGraph, todo son clases:

```python
# Ejemplo real de LangChain - NO ejecutar, solo ilustrativo
from langchain_openai import ChatOpenAI

# ChatOpenAI es una CLASE, llm es un OBJETO (instancia)
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

# .invoke() es un MÉTODO del objeto
response = llm.invoke("Hola")
```

Si no entiendes clases, no entenderás el código que escribes.

## Conceptos fundamentales

| Concepto | Descripción | En agentes |
|----------|-------------|------------|
| **Clase** | Plantilla/molde | `ChatOpenAI`, `BaseTool` |
| **Objeto** | Instancia de clase | `llm = ChatOpenAI()` |
| **Atributo** | Datos del objeto | `llm.model_name` |
| **Método** | Funciones del objeto | `llm.invoke()` |
| **Constructor** | Inicializa el objeto | `def __init__(self):` |

In [ ]:
# TU PRIMERA CLASE: Un "Tool" básico

class Calculator:
    """
    Una herramienta de cálculo simple.
    
    En LangChain, las herramientas (tools) son clases.
    Esta es una versión simplificada del patrón.
    """
    
    def __init__(self, name="calculator"):
        """
        Constructor: se ejecuta al crear el objeto.
        
        'self' = referencia al propio objeto
        """
        # Atributos - datos del objeto
        self.name = name
        self.operations_count = 0
    
    def add(self, a, b):
        """Método: suma dos números."""
        self.operations_count += 1  # Usamos self para acceder a atributos
        return a + b
    
    def multiply(self, a, b):
        """Método: multiplica dos números."""
        self.operations_count += 1
        return a * b
    
    def get_stats(self):
        """Método: devuelve estadísticas."""
        return f"Tool '{self.name}' ha ejecutado {self.operations_count} operaciones"


# Crear un objeto (instancia) de la clase
calc = Calculator(name="mi_calculadora")

# Usar los métodos
print(calc.add(5, 3))
print(calc.multiply(4, 7))
print(calc.add(10, 20))

# Ver estadísticas
print(calc.get_stats())

# Acceder a atributos directamente
print(f"Nombre del tool: {calc.name}")

## Herencia: Reutilizar y extender

La **herencia** permite crear clases basadas en otras. Es el patrón de LangChain:

```python
# Patrón real de LangChain (ilustrativo)
from langchain.tools import BaseTool

class MiTool(BaseTool):  # Hereda de BaseTool
    name = "mi_tool"
    description = "Hace algo útil"
    
    def _run(self, query):  # Implementa método requerido
        return f"Procesando: {query}"
```

Veamos cómo funciona la herencia:

In [ ]:
# HERENCIA: El patrón de LangChain

class BaseTool:
    """
    Clase base para herramientas.
    Define la estructura que deben seguir todas las tools.
    """
    
    name = "base_tool"
    description = "Tool base"
    
    def __init__(self):
        self.call_count = 0
    
    def run(self, input_text):
        """Método público que los usuarios llaman."""
        self.call_count += 1
        # Llama al método que las subclases deben implementar
        return self._run(input_text)
    
    def _run(self, input_text):
        """Método que las subclases DEBEN sobrescribir."""
        raise NotImplementedError("Subclases deben implementar _run()")


class SearchTool(BaseTool):
    """Tool de búsqueda - hereda de BaseTool."""
    
    name = "search"
    description = "Busca información en internet"
    
    def __init__(self, search_engine="google"):
        super().__init__()  # Llama al __init__ del padre
        self.search_engine = search_engine
    
    def _run(self, query):
        """Implementación específica de búsqueda."""
        # Simulamos una búsqueda
        return f"[{self.search_engine}] Resultados para: '{query}'"


class WeatherTool(BaseTool):
    """Tool del clima - hereda de BaseTool."""
    
    name = "weather"
    description = "Obtiene el clima de una ciudad"
    
    def _run(self, city):
        """Implementación específica del clima."""
        # Simulamos obtener clima
        return f"Clima en {city}: 22°C, soleado"


# Crear instancias de las tools
search = SearchTool(search_engine="duckduckgo")
weather = WeatherTool()

# Usar las tools
print(search.run("restaurantes en Madrid"))
print(weather.run("Barcelona"))

# Verificar herencia
print(f"\n¿search es BaseTool? {isinstance(search, BaseTool)}")
print(f"Nombre: {search.name}, Descripción: {search.description}")

---
# PARTE 2: TYPE HINTS (Anotaciones de Tipos)

## ¿Qué son los Type Hints?
Son **anotaciones opcionales** que indican qué tipos de datos espera una función o variable.

```python
# Sin type hints
def greet(name):
    return f"Hola {name}"

# Con type hints
def greet(name: str) -> str:
    return f"Hola {name}"
```

## ¿Por qué son CRÍTICOS para agentes?
LangGraph **requiere** types para definir el estado:

```python
# Código REAL de LangGraph
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    current_step: str
```

Sin entender types, no puedes definir el estado de tu agente.

## Tipos básicos de `typing`

| Tipo | Uso | Ejemplo |
|------|-----|---------|
| `str`, `int`, `float`, `bool` | Tipos primitivos | `name: str` |
| `list[T]` | Lista de tipo T | `names: list[str]` |
| `dict[K, V]` | Diccionario | `data: dict[str, int]` |
| `Optional[T]` | T o None | `age: Optional[int]` |
| `Union[A, B]` | A o B | `id: Union[str, int]` |
| `Any` | Cualquier tipo | `data: Any` |
| `TypedDict` | Dict con estructura fija | `class State(TypedDict):` |
| `Callable` | Función | `func: Callable[[int], str]` |

In [ ]:
# TYPE HINTS BÁSICOS

from typing import Optional, Union

# Función sin type hints (ambigua)
def process_bad(data, count):
    return data * count

# Función con type hints (clara)
def process_good(data: str, count: int) -> str:
    """
    Repite un string 'count' veces.
    
    Args:
        data: El texto a repetir
        count: Número de repeticiones
        
    Returns:
        El texto repetido
    """
    return data * count


# El IDE y herramientas pueden verificar tipos
result = process_good("Hola ", 3)
print(result)


# Optional: puede ser el tipo indicado O None
def find_user(user_id: int) -> Optional[dict]:
    """Busca un usuario. Devuelve None si no existe."""
    users = {1: {"name": "Ana"}, 2: {"name": "Luis"}}
    return users.get(user_id)  # .get() devuelve None si no existe

user = find_user(1)
print(f"Usuario 1: {user}")

user = find_user(999)
print(f"Usuario 999: {user}")


# Union: puede ser uno u otro tipo
def get_id(item: dict) -> Union[str, int]:
    """Devuelve el ID, que puede ser string o int."""
    return item.get("id", 0)

print(f"\nID como int: {get_id({'id': 123})}")
print(f"ID como str: {get_id({'id': 'abc-123'})}")

In [ ]:
# TypedDict: EL PATRÓN DE LANGGRAPH

from typing import TypedDict

# TypedDict define un diccionario con estructura fija
# Este es EXACTAMENTE el patrón de LangGraph

class AgentState(TypedDict):
    """
    Estado del agente - estructura fija y tipada.
    
    En LangGraph, SIEMPRE defines tu estado así.
    """
    messages: list          # Lista de mensajes
    current_step: str       # Paso actual del flujo
    user_info: dict         # Información del usuario
    tool_results: list      # Resultados de tools


# Crear un estado válido
state: AgentState = {
    "messages": [
        {"role": "user", "content": "Hola"},
        {"role": "assistant", "content": "¡Hola! ¿En qué puedo ayudarte?"}
    ],
    "current_step": "waiting_input",
    "user_info": {"name": "María", "premium": True},
    "tool_results": []
}

print("=== AGENT STATE ===")
print(f"Mensajes: {len(state['messages'])}")
print(f"Paso actual: {state['current_step']}")
print(f"Usuario: {state['user_info']['name']}")


# Función que opera sobre el estado (patrón de nodos en LangGraph)
def process_message(state: AgentState) -> AgentState:
    """
    Un nodo de LangGraph recibe el estado y devuelve el estado modificado.
    """
    # Agregar un mensaje
    new_message = {"role": "assistant", "content": "Procesando tu solicitud..."}
    state["messages"].append(new_message)
    state["current_step"] = "processing"
    return state


# Simular un nodo procesando el estado
updated_state = process_message(state)
print(f"\nDespués de procesar:")
print(f"Mensajes: {len(updated_state['messages'])}")
print(f"Paso actual: {updated_state['current_step']}")

---
# PARTE 3: PYDANTIC - Structured Output

## ¿Qué es Pydantic?
**Pydantic** es una librería para definir modelos de datos con validación automática.

## ¿Por qué es ESENCIAL para agentes?
Cuando quieres que el LLM devuelva datos estructurados (no solo texto), usas Pydantic:

```python
# Ejemplo REAL de LangChain
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class MovieReview(BaseModel):
    title: str
    rating: int
    summary: str

llm = ChatOpenAI()
structured_llm = llm.with_structured_output(MovieReview)

# Ahora el LLM devuelve un objeto MovieReview, no texto
result = structured_llm.invoke("Analiza la película Inception")
print(result.title)   # "Inception"
print(result.rating)  # 9
```

## Pydantic vs TypedDict

| Característica | TypedDict | Pydantic |
|---------------|-----------|----------|
| Validación | ❌ No valida | ✅ Valida tipos |
| Valores por defecto | ❌ No | ✅ Sí |
| Serialización JSON | Manual | ✅ Automática |
| Uso en LangGraph | Estado del grafo | Structured output |

**Regla práctica:**
- **TypedDict** → Para el State de LangGraph
- **Pydantic** → Para structured output del LLM

In [ ]:
# PYDANTIC BÁSICO
# Primero instala: pip install pydantic

from pydantic import BaseModel, Field
from typing import Optional, List

# Definir un modelo Pydantic
class UserQuery(BaseModel):
    """
    Modelo para representar una consulta del usuario.
    
    El LLM puede devolver datos en este formato exacto.
    """
    intent: str = Field(description="La intención del usuario: question, command, greeting")
    entities: List[str] = Field(default=[], description="Entidades mencionadas")
    confidence: float = Field(ge=0, le=1, description="Confianza de 0 a 1")
    original_text: str = Field(description="El texto original")


# Crear instancias - Pydantic valida automáticamente
query1 = UserQuery(
    intent="question",
    entities=["clima", "Madrid"],
    confidence=0.95,
    original_text="¿Cómo está el clima en Madrid?"
)

print("=== QUERY VÁLIDA ===")
print(f"Intent: {query1.intent}")
print(f"Entities: {query1.entities}")
print(f"Confidence: {query1.confidence}")

# Convertir a diccionario (útil para JSON)
print(f"\nComo dict: {query1.model_dump()}")

# Convertir a JSON
print(f"\nComo JSON: {query1.model_dump_json()}")

### Validación automática

Pydantic valida los datos automáticamente. Si pasas datos inválidos, lanza un error claro:

In [ ]:
# VALIDACIÓN AUTOMÁTICA DE PYDANTIC

from pydantic import BaseModel, Field, ValidationError

class ToolCall(BaseModel):
    """Representa una llamada a herramienta."""
    tool_name: str
    arguments: dict
    priority: int = Field(ge=1, le=5, default=3)  # Entre 1 y 5


# Crear con datos válidos
valid_call = ToolCall(
    tool_name="search",
    arguments={"query": "restaurantes"},
    priority=2
)
print(f"Válido: {valid_call}")

# Intentar crear con datos inválidos
print("\n=== VALIDACIÓN DE ERRORES ===")
try:
    invalid_call = ToolCall(
        tool_name="search",
        arguments={"query": "test"},
        priority=10  # ¡Inválido! Debe ser <= 5
    )
except ValidationError as e:
    print(f"Error de validación:\n{e}")

### Modelos anidados (composición)

Puedes combinar modelos Pydantic para estructuras complejas:

In [ ]:
# MODELOS ANIDADOS - Estructuras complejas

from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum


# Enum para valores predefinidos
class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


class Step(BaseModel):
    """Un paso individual del plan."""
    action: str = Field(description="Qué hacer")
    tool: Optional[str] = Field(default=None, description="Tool a usar")
    expected_output: str = Field(description="Qué esperamos obtener")


class AgentPlan(BaseModel):
    """
    Plan estructurado que el agente genera.
    
    Este es el tipo de structured output que puedes
    pedirle a un LLM que genere.
    """
    goal: str = Field(description="Objetivo principal")
    priority: Priority = Field(default=Priority.MEDIUM)
    steps: List[Step] = Field(description="Pasos a ejecutar")
    estimated_time: Optional[str] = Field(default=None)


# Crear un plan estructurado
plan = AgentPlan(
    goal="Encontrar vuelos baratos a París",
    priority=Priority.HIGH,
    steps=[
        Step(
            action="Buscar vuelos disponibles",
            tool="flight_search",
            expected_output="Lista de vuelos con precios"
        ),
        Step(
            action="Filtrar por precio",
            tool=None,
            expected_output="Top 3 vuelos más baratos"
        ),
        Step(
            action="Verificar disponibilidad",
            tool="booking_check",
            expected_output="Confirmación de disponibilidad"
        )
    ],
    estimated_time="5 minutos"
)

print("=== PLAN DEL AGENTE ===")
print(f"Objetivo: {plan.goal}")
print(f"Prioridad: {plan.priority.value}")
print(f"Tiempo estimado: {plan.estimated_time}")
print(f"\nPasos:")
for i, step in enumerate(plan.steps, 1):
    tool_info = f" [Tool: {step.tool}]" if step.tool else ""
    print(f"  {i}. {step.action}{tool_info}")

# Serializar a JSON (listo para guardar o enviar)
print(f"\n=== COMO JSON ===")
print(plan.model_dump_json(indent=2))

---
# PARTE 4: DECORADORES

## ¿Qué es un decorador?
Un **decorador** es una función que "envuelve" otra función para modificar su comportamiento.

```python
@mi_decorador
def mi_funcion():
    pass
```

Es equivalente a:
```python
def mi_funcion():
    pass
mi_funcion = mi_decorador(mi_funcion)
```

## ¿Por qué son importantes para agentes?
LangChain usa decoradores constantemente:

```python
from langchain_core.tools import tool

@tool
def search(query: str) -> str:
    """Busca información."""  # ← El docstring es la descripción
    return f"Resultados para: {query}"
```

El decorador `@tool` convierte tu función en una herramienta que el agente puede usar.

In [ ]:
# DECORADORES BÁSICOS

# Un decorador es una función que recibe una función y devuelve otra función

def log_calls(func):
    """
    Decorador que imprime cuándo se llama una función.
    """
    def wrapper(*args, **kwargs):
        print(f"📞 Llamando a: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"✅ {func.__name__} completado")
        return result
    return wrapper


# Usar el decorador
@log_calls
def greet(name: str) -> str:
    """Saluda a alguien."""
    return f"¡Hola, {name}!"


@log_calls  
def add(a: int, b: int) -> int:
    """Suma dos números."""
    return a + b


# Llamar a las funciones decoradas
print(greet("María"))
print()
print(f"Resultado: {add(5, 3)}")

In [ ]:
# SIMULANDO EL DECORADOR @tool DE LANGCHAIN

from functools import wraps
from typing import Callable

def tool(func: Callable) -> Callable:
    """
    Decorador que convierte una función en una "herramienta".
    
    Similar a @tool de LangChain (versión simplificada).
    """
    @wraps(func)  # Preserva el nombre y docstring original
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    
    # Añadir metadatos (como hace LangChain)
    wrapper.name = func.__name__
    wrapper.description = func.__doc__ or "Sin descripción"
    wrapper.is_tool = True
    
    return wrapper


# Definir herramientas con el decorador
@tool
def search(query: str) -> str:
    """Busca información en internet sobre el tema indicado."""
    return f"[Búsqueda] Resultados para: '{query}'"


@tool
def get_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad específica."""
    return f"[Clima] {city}: 22°C, soleado"


@tool
def calculate(expression: str) -> str:
    """Evalúa una expresión matemática y devuelve el resultado."""
    try:
        result = eval(expression)
        return f"[Cálculo] {expression} = {result}"
    except:
        return "[Error] Expresión inválida"


# Ver los metadatos que añadió el decorador
print("=== HERRAMIENTAS DISPONIBLES ===\n")
tools = [search, get_weather, calculate]

for t in tools:
    print(f"🔧 {t.name}")
    print(f"   Descripción: {t.description}")
    print(f"   Es tool: {t.is_tool}")
    print()

In [ ]:
# USAR LAS HERRAMIENTAS

# Simular cómo un agente usaría las tools

def agent_execute(tool_name: str, argument: str, available_tools: list) -> str:
    """
    Simula un agente ejecutando una herramienta.
    
    En LangChain/LangGraph, el LLM decide qué tool usar
    y el framework la ejecuta.
    """
    # Buscar la tool por nombre
    for t in available_tools:
        if t.name == tool_name:
            print(f"🤖 Agente ejecutando: {tool_name}('{argument}')")
            result = t(argument)
            return result
    
    return f"[Error] Tool '{tool_name}' no encontrada"


# El agente "decide" usar herramientas
tools = [search, get_weather, calculate]

print("=== AGENTE EJECUTANDO TOOLS ===\n")

# Simulando decisiones del agente
result1 = agent_execute("search", "mejores restaurantes Madrid", tools)
print(f"Resultado: {result1}\n")

result2 = agent_execute("get_weather", "Barcelona", tools)
print(f"Resultado: {result2}\n")

result3 = agent_execute("calculate", "15 * 24 + 100", tools)
print(f"Resultado: {result3}")

---
# PARTE 5: INTEGRACIÓN - Componente Completo

Ahora combinemos todo lo aprendido para crear un componente estilo LangChain:

In [ ]:
# INTEGRACIÓN COMPLETA: ToolRegistry con todo lo aprendido

from typing import TypedDict, Callable, Optional, List
from pydantic import BaseModel, Field
from functools import wraps
from datetime import datetime


# ========== MODELOS PYDANTIC ==========

class ToolSpec(BaseModel):
    """Especificación de una herramienta."""
    name: str
    description: str
    category: str = "general"


class ToolResult(BaseModel):
    """Resultado de ejecutar una herramienta."""
    tool_name: str
    input: str
    output: str
    success: bool
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())


# ========== DECORADOR @tool ==========

def tool(category: str = "general"):
    """
    Decorador que registra una función como herramienta.
    
    Uso:
        @tool(category="search")
        def my_search(query: str) -> str:
            '''Descripción de la tool.'''
            return "resultado"
    """
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        
        wrapper.name = func.__name__
        wrapper.description = func.__doc__ or "Sin descripción"
        wrapper.category = category
        wrapper.is_tool = True
        
        return wrapper
    return decorator


# ========== CLASE PRINCIPAL ==========

class ToolRegistry:
    """
    Registro de herramientas disponibles.
    
    Patrón similar a cómo LangChain gestiona tools.
    Usa POO, type hints y composición.
    """
    
    def __init__(self):
        self.tools: dict[str, Callable] = {}
        self.execution_history: List[ToolResult] = []
    
    def register(self, tool_func: Callable) -> None:
        """Registra una herramienta."""
        if not getattr(tool_func, 'is_tool', False):
            raise ValueError(f"{tool_func.__name__} no está decorado con @tool")
        
        self.tools[tool_func.name] = tool_func
        print(f"✅ Registrado: {tool_func.name} [{tool_func.category}]")
    
    def list_tools(self) -> List[ToolSpec]:
        """Lista todas las herramientas disponibles."""
        return [
            ToolSpec(
                name=t.name,
                description=t.description,
                category=t.category
            )
            for t in self.tools.values()
        ]
    
    def execute(self, tool_name: str, input_text: str) -> ToolResult:
        """Ejecuta una herramienta y registra el resultado."""
        if tool_name not in self.tools:
            return ToolResult(
                tool_name=tool_name,
                input=input_text,
                output=f"Error: Tool '{tool_name}' no existe",
                success=False
            )
        
        tool_func = self.tools[tool_name]
        
        try:
            output = tool_func(input_text)
            result = ToolResult(
                tool_name=tool_name,
                input=input_text,
                output=output,
                success=True
            )
        except Exception as e:
            result = ToolResult(
                tool_name=tool_name,
                input=input_text,
                output=f"Error: {str(e)}",
                success=False
            )
        
        self.execution_history.append(result)
        return result
    
    def get_history(self) -> List[ToolResult]:
        """Devuelve el historial de ejecuciones."""
        return self.execution_history


# ========== DEFINIR HERRAMIENTAS ==========

@tool(category="search")
def web_search(query: str) -> str:
    """Busca información en la web."""
    return f"Resultados de búsqueda para: '{query}'"


@tool(category="weather")
def get_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad."""
    return f"Clima en {city}: 24°C, parcialmente nublado"


@tool(category="math")
def calculator(expression: str) -> str:
    """Evalúa expresiones matemáticas."""
    result = eval(expression)
    return f"{expression} = {result}"


# ========== USO ==========

# Crear registro
registry = ToolRegistry()

# Registrar herramientas
registry.register(web_search)
registry.register(get_weather)
registry.register(calculator)

print("\n=== HERRAMIENTAS DISPONIBLES ===")
for spec in registry.list_tools():
    print(f"  🔧 {spec.name} ({spec.category}): {spec.description}")

In [ ]:
# EJECUTAR HERRAMIENTAS

print("=== EJECUTANDO HERRAMIENTAS ===\n")

# Ejecutar varias herramientas
results = [
    registry.execute("web_search", "restaurantes Madrid"),
    registry.execute("get_weather", "Barcelona"),
    registry.execute("calculator", "125 * 8"),
    registry.execute("tool_inexistente", "test")  # Error controlado
]

for result in results:
    status = "✅" if result.success else "❌"
    print(f"{status} {result.tool_name}('{result.input}')")
    print(f"   → {result.output}")
    print()

# Ver historial
print("=== HISTORIAL DE EJECUCIONES ===")
print(f"Total de ejecuciones: {len(registry.get_history())}")
print(f"Exitosas: {sum(1 for r in registry.get_history() if r.success)}")
print(f"Fallidas: {sum(1 for r in registry.get_history() if not r.success)}")

---
# 📝 Resumen del Notebook 2

## Lo que aprendiste

| Concepto | Para qué lo usarás |
|----------|-------------------|
| **Clases y POO** | Entender `ChatOpenAI()`, `BaseTool`, todos los componentes |
| **Herencia** | Crear tus propias tools extendiendo `BaseTool` |
| **Type Hints** | Definir `AgentState(TypedDict)` en LangGraph |
| **TypedDict** | El estado de tu grafo en LangGraph |
| **Pydantic** | Structured output - que el LLM devuelva datos tipados |
| **Decoradores** | Usar `@tool` para crear herramientas |

## Código real que ahora entiendes

```python
# LangGraph - Estado con TypedDict
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    
# LangChain - Tool con decorador
@tool
def search(query: str) -> str:
    """Busca en internet."""
    return results

# LangChain - Structured output con Pydantic
class Answer(BaseModel):
    response: str
    sources: list[str]
    
structured_llm = llm.with_structured_output(Answer)
```
